# Parallel Hull-White Calibration with ParFun + QuantLib

The **Hull-White model** is a short-rate model widely used for interest-rate derivatives pricing.
Calibrating it involves fitting model parameters to a grid of swaption prices using
a tree-based pricing engine.

When multiple yield curves need calibration (e.g. different currencies or scenarios),
each calibration is independent and can run in parallel.

ParFun parallelises this with minimal code changes.

## Program Structure

```mermaid
flowchart TB
    subgraph seq["Sequential"]
        direction TB
        S1["Curve A calibrate"] --> S2["Curve B calibrate"]
        S2 --> S3["Curve C calibrate"] --> S4["..."]
        S4 --> S5["Curve H calibrate"]
    end

    subgraph par["Parallel 4 workers"]
        direction TB
        subgraph W1["Worker 1"]
            A1["Curve A calibrate"]
            A1b["Curve B calibrate"]
        end
        subgraph W2["Worker 2"]
            A2["Curve C calibrate"]
            A2b["Curve D calibrate"]
        end
        subgraph W3["Worker 3"]
            A3["Curve E calibrate"]
            A3b["Curve F calibrate"]
        end
        subgraph W4["Worker 4"]
            A4["Curve G calibrate"]
            A4b["Curve H calibrate"]
        end
        W1 & W2 & W3 & W4 --> C1(["concat results"])
    end
```

# Imports and Environment

In [1]:
import time

import QuantLib as ql

import parfun as pf

# Calibrate a single Hull-White model for one yield curve against a swaption grid.

In [2]:
def calibrate_hw_once(curve_data):
    """Calibrate a Hull-White model for a yield curve.
    curve_data = (curve_id, base_rate)
    Returns (curve_id, calibrated_params)."""
    curve_id, base_rate = curve_data
    today = ql.Date(1, 1, 2025)
    ql.Settings.instance().evaluationDate = today

    tenors_months = [0, 6, 12, 24, 36, 60, 84, 120, 180, 240, 360, 480]
    dates = [today + ql.Period(m, ql.Months) for m in tenors_months]
    rates = [base_rate + i * 0.002 for i in range(len(tenors_months))]

    curve = ql.ZeroCurve(dates, rates, ql.Actual365Fixed())
    curve_handle = ql.YieldTermStructureHandle(curve)
    index = ql.Euribor6M(curve_handle)

    model = ql.HullWhite(curve_handle)
    engine = ql.TreeSwaptionEngine(model, 150)

    helpers = []
    for expiry_years in [1, 2, 3, 5, 7, 10, 15, 20]:
        for swap_tenor in [2, 5, 10, 15]:
            h = ql.SwaptionHelper(
                ql.Period(expiry_years, ql.Years),
                ql.Period(swap_tenor, ql.Years),
                ql.QuoteHandle(ql.SimpleQuote(0.0055)),
                index, index.tenor(), index.dayCounter(), index.dayCounter(),
                curve_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(
        helpers, ql.LevenbergMarquardt(),
        ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8),
    )
    return curve_id, list(model.params())

# Function to Parallelize

In [3]:
def calibrate_hw(curve_data_list):
    """Calibrate Hull-White for each yield curve."""
    return [calibrate_hw_once(d) for d in curve_data_list]

# Parallelizing with ParFun

The only change needed is wrapping `calibrate_hw` with a `@pf.parfun` decorator.
The parallel version calls the **same function** — no logic is duplicated:

```diff
+ @pf.parfun(
+     split=pf.per_argument(curve_data_list=pf.py_list.by_chunk),
+     combine_with=pf.py_list.concat,
+     fixed_partition_size=1,
+ )
  def calibrate_hw_w_parfun(curve_data_list):
+     return calibrate_hw(curve_data_list)
  ...
+ with pf.set_parallel_backend_context("scaler_local", n_workers=4):
+     result = calibrate_hw_w_parfun(curves)
```

In [4]:
@pf.parfun(
    split=pf.per_argument(curve_data_list=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def calibrate_hw_w_parfun(curve_data_list):
    return calibrate_hw(curve_data_list)

In [5]:
curves = [
    ("curve_A", 0.010),
    ("curve_B", 0.015),
    ("curve_C", 0.005),
    ("curve_D", 0.020),
    ("curve_E", 0.008),
    ("curve_F", 0.012),
    ("curve_G", 0.018),
    ("curve_H", 0.003),
]

start = time.time()
results_seq = calibrate_hw(curves)
seq_time = time.time() - start

start = time.time()
with pf.set_parallel_backend_context("scaler_local", n_workers=4):
    results_par = calibrate_hw_w_parfun(curves)
par_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time / par_time:.2f}x")

[INFO]2026-03-28 03:09:18+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 03:09:18+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:47537
[INFO]2026-03-28 03:09:18+0800: ObjectStorageServer: started
[INFO]2026-03-28 03:09:18+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 03:09:18+0800: use event loop: builtin
[INFO]2026-03-28 03:09:18+0800: ConfigController: scheduler_address = tcp://127.0.0.1:38433
[INFO]2026-03-28 03:09:18+0800: ConfigController: object_storage_address = tcp://127.0.0.1:47537
[INFO]2026-03-28 03:09:18+0800: ConfigController: monitor_address = None
[INFO]2026-03-28 03:09:18+0800: ConfigController: protected = True
[INFO]2026-03-28 03:09:18+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-28 03:09:18+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-28 03:09:18+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-28 03:09:18+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-28 03: